# Qwen2.5-VL 3B Annotations
- Using the Qwen2.5-VL 3B model to generate ground truth OCR data for SROIEv2 dataset.
- Model used: Qwen2.5-VL 3B

In [ ]:
#!pip install qwen_vl_utils
#!pip install flash_attn

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
import torch
import glob
import os

## 1. Load model

In [ ]:
model_id = 'Qwen/Qwen2.5-VL-3B-Instruct'

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    # attn_implementation="flash_attention_2", # Removed due to runtime incompatibility
    device_map="auto"
)

# Load processor
processor = AutoProcessor.from_pretrained(model_id)

## 2. Load dataset

In [ ]:
!mkdir /root/.kaggle
!echo '{"username":"USERNAME","key":"API_KEY"}' > /root/.kaggle/kaggle.json
!kaggle datasets download -d urbikn/sroie-datasetv2
!chmod 600 /root/.kaggle/kaggle.json
!unzip -q sroie-datasetv2.zip -d sroie_v2

In [ ]:
image_paths_list = [
    './sroie_v2/SROIE2019/train/img/*.jpg', # Train
    './sroie_v2/SROIE2019/test/img/*.jpg' # Test
]
out_dir_list = [
    './annots/qwen2_5_vl_3b_annots/train_annots', #Train
    './annots/qwen2_5_vl_3b_annots/test_annots' # Test
]

## 3. Batch Annotation

In [ ]:
class BatchedDataset(Dataset):
    def __init__(self, all_images):
        self.all_images = all_images

    def __len__(self):
        return len(self.all_images)

    def __getitem__(self, idx):
        return self.all_images[idx]

def batch_infer(messages):
    # Preparation for inference
    texts = [
            processor.apply_chat_template(
            msg, tokenize=False, add_generation_prompt=True
        )
        for msg in messages
    ]

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )
    inputs = inputs.to("cuda")

    # Inference: Generation of the output
    generated_ids = model.generate(**inputs, max_new_tokens=1024)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    # print(output_text)
    return output_text

def process_images_in_batches(image_paths_list, out_dir_list, batch_size, processor, model, user_message):
    """
    Processes images in batches for OCR and saves the results.

    Args:
        image_paths_list (list): A list of glob patterns for image files.
        out_dir_list (list): A list of output directories corresponding to image_paths_list.
        batch_size (int): The number of images to process in each batch.
        processor (AutoProcessor): The processor for the Qwen2.5-VL model.
        model (Qwen2_5_VLForConditionalGeneration): The loaded Qwen2.5-VL model.
    """
    for image_path_pattern, out_dir in zip(image_paths_list, out_dir_list):
        # Get all image paths for the current dataset split (train or test)
        all_images = glob.glob(image_path_pattern)
        # Create the output directory if it doesn't exist
        os.makedirs(out_dir, exist_ok=True)

        # Create a custom dataset from the image paths
        custom_dataset = BatchedDataset(all_images)

        # Create a DataLoader for batching the images
        batch_dl = DataLoader(custom_dataset, batch_size=batch_size, shuffle=False)

        print(f'####### Processing images from {image_path_pattern} #######')

        # Iterate through batches of images
        for batch in tqdm(batch_dl, total=len(batch_dl)):
            messages = []

            # Create a message for each image in the batch
            for image_path in batch:
                message = [
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "image": image_path,
                                "resized_height": 768,
                                "resized_width": 512,
                            },
                            {"type": "text", "text": user_message},
                        ],
                    }
                ]
                messages.append(message)

            # Perform batch inference on the messages
            texts = batch_infer(messages)

            # Save the generated text for each image
            for text, image_path in zip(texts, batch):
                # print(text)
                # Create the output filename based on the image filename
                output_filename = os.path.join(out_dir, image_path.split(os.path.sep)[-1].split('.jpg')[0]+'.txt')
                with open(output_filename, 'w') as f:
                    f.write(text)

In [ ]:
# Example usage
batch_size = 6
device = 'cuda'
user_message = 'Give the OCR text from this image and nothing else. Read the bill image carefully and extract all the information. Capture every possible detail including Restaurant name, Address, Bill number, order number, date, time, Itemized list, total bill amount, GST number, Website, email, and footer notes.'
process_images_in_batches(image_paths_list, out_dir_list, batch_size, 
                          processor, model, user_message)